# SNC — Train the Query-Conditioned Separator

Trains the **scalable** separation model: a U-Net that is told *which*
class to extract and emits a single mask for it. It handles all 50
ESC-50 classes (and can extend to more datasets) with one architecture.

Mixtures are generated **on the fly** during training — there is no
multi-gigabyte dataset file.

## Pipeline

| Stage | Script | Output |
|------|--------|--------|
| 0 | (cell) | Download ESC-50 to Drive if missing |
| 1 | `train_conditioned_separator.py` | `best_conditioned_separator.h5` |
| 2 | `evaluate_conditioned_separator.py` | SI-SDR results table |
| 3 | `conditioned_selective_separation.py` | Remove sounds from your own file |

## Before running

**Runtime → Change runtime type → GPU.** Pick the **T4** GPU — newer
GPUs (A100 / L4 / Blackwell) can fail with `CUDA_ERROR_INVALID_PTX`
because the installed TensorFlow has no kernels for them. T4 is fully
supported and trains this model fine.

## 1. Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
BRANCH = 'feature/conditioned-separation'

# Private repo? Add a Colab Secret named GITHUB_TOKEN (key icon, scope: repo).
import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    clone_url = f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('Using GITHUB_TOKEN from Colab Secrets.')
else:
    clone_url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('No GITHUB_TOKEN secret found — attempting anonymous clone.')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
result = subprocess.run(
    ['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(
        f'git clone failed (exit {result.returncode}). '
        'If the repo is private, add a GITHUB_TOKEN secret in the left sidebar.'
    )
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-5'], check=True)

## 2. Symlink data and saved_models to Drive

In [ ]:
drive_data = Path(DRIVE_ROOT) / 'data'
drive_models = Path(DRIVE_ROOT) / 'saved_models'
drive_data.mkdir(parents=True, exist_ok=True)
drive_models.mkdir(parents=True, exist_ok=True)

for local, target in [(REPO_DIR / 'data', drive_data),
                       (REPO_DIR / 'saved_models', drive_models)]:
    if local.is_symlink() or local.exists():
        local.unlink() if local.is_symlink() else shutil.rmtree(local)
    local.symlink_to(target)
    print(f'{local} -> {target}')

## Stage 0 — Download ESC-50 (only if missing)

In [ ]:
import zipfile, urllib.request

archive_dir = REPO_DIR / 'data' / 'raw' / 'archive'
csv_path = archive_dir / 'esc50.csv'
wav_dir = archive_dir / 'audio' / 'audio'

if csv_path.exists() and wav_dir.exists() and len(list(wav_dir.glob('*.wav'))) >= 2000:
    print(f'ESC-50 already present — {len(list(wav_dir.glob("*.wav")))} WAV files.')
else:
    archive_dir.mkdir(parents=True, exist_ok=True)
    zip_path = Path('/content/esc50.zip')
    url = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'
    print('Downloading ESC-50 (~600 MB on Colab network)...')
    urllib.request.urlretrieve(url, zip_path)

    extract_dir = Path('/content/esc50_extracted')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    src_root = extract_dir / 'ESC-50-master'

    shutil.copy(src_root / 'meta' / 'esc50.csv', csv_path)
    wav_dir.mkdir(parents=True, exist_ok=True)
    print('Copying WAV files to Drive...')
    for wav in (src_root / 'audio').glob('*.wav'):
        shutil.copy(wav, wav_dir / wav.name)

    zip_path.unlink()
    shutil.rmtree(extract_dir)
    print(f'Done — {len(list(wav_dir.glob("*.wav")))} WAV files.')

assert csv_path.exists() and wav_dir.exists()

## 3. Install dependencies

In [ ]:
!pip install -q librosa==0.11.0 soundfile scikit-learn pandas
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPUs       :', tf.config.list_physical_devices('GPU'))

## Stage 1 — Train the conditioned separator

Mixtures are generated on the fly — no dataset file. On a T4 this is
roughly 45-90 minutes depending on `epochs` / `steps_per_epoch`. The
best checkpoint is written to Drive as it improves.

In [ ]:
!python src/model_training/train_conditioned_separator.py

## Stage 2 — Evaluate (SI-SDR)

In [ ]:
!python src/model_training/evaluate_conditioned_separator.py

## Stage 3 — Remove sounds from your own file

Upload an audio file, then set a gain per class: `0.0` removes a
sound, `1.0` keeps it, `0.5` halves it. Classes you do not list are
left untouched.

In [ ]:
from google.colab import files

uploaded = files.upload()
INPUT_FILE = '/content/' + list(uploaded.keys())[0]
print('Uploaded:', INPUT_FILE)

In [ ]:
import sys, json
sys.path.insert(0, str(REPO_DIR / 'src' / 'application'))
from conditioned_selective_separation import ConditionedSelectiveSeparator

models = REPO_DIR / 'saved_models' / 'separation_models'
print('Available classes:', json.load(open(models / 'conditioned_class_names.json')))

# --- EDIT: gain per class. Omit a class to keep it (gain 1.0). ---
GAINS = {'engine': 0.0, 'wind': 0.0}
# -----------------------------------------------------------------

separator = ConditionedSelectiveSeparator(
    model_path=models / 'best_conditioned_separator.h5',
    class_names_path=models / 'conditioned_class_names.json',
)
separator.process(
    input_path=Path(INPUT_FILE),
    output_path=Path('/content/conditioned_output.wav'),
    gains=GAINS,
)

In [ ]:
from IPython.display import Audio, display
import librosa

orig, _ = librosa.load(INPUT_FILE, sr=16000, mono=True)
result, _ = librosa.load('/content/conditioned_output.wav', sr=16000, mono=True)
print('Original:')
display(Audio(orig, rate=16000))
print('After selective removal:')
display(Audio(result, rate=16000))
files.download('/content/conditioned_output.wav')